# ⚡ Istovremeni radni tokovi agenata s Microsoft Foundry (Python)

## 📋 Napredni vodič za paralelnu obradu

Ovaj bilježnik demonstrira **uzorke paralelnih radnih tokova** pomoću Microsoft Agent Frameworka. Naučit ćete kako izgraditi visokoučinkovite radne tokove paralelne obrade u kojima više AI agenata istovremeno izvršava zadatke, značajno poboljšavajući propusnost i omogućujući složene višedretvene poslovne procese.

> **Napomena o migraciji:** Ovaj primjer je prethodno referencirao GitHub modele. GitHub modeli se ukidaju (krajem srpnja 2026), stoga sada koristi **Microsoft Foundry** preko `FoundryChatClient` koji cilja Azure OpenAI **Responses API**.

## 🎯 Ciljevi učenja

### 🚀 **Osnove istovremene obrade**
- **Istovremeno izvođenje agenata**: Pokrenite više agenata istovremeno za maksimalnu učinkovitost
- **Orkestracija radnih tokova**: Koordinirajte istovremene operacije uz održavanje konzistencije podataka
- **Optimizacija performansi**: Postignite značajno ubrzanje kroz paralelnu obradu
- **Upravljanje resursima**: Efikasno koristite AI modele tijekom istovremenih operacija

### 🏗️ **Napredni obrasci konkurentnosti**
- **Fork-Join obrada**: Podijelite rad među više agenata i objedinite rezultate
- **Paralelizam cijevi**: Preklapajuće faze izvođenja za kontinuiranu propusnost
- **Ravnoteža opterećenja**: Ravnomjerno raspodijelite rad na dostupne resurse agenata
- **Točke sinkronizacije**: Koordinirajte istovremene agente u ključnim fazama radnog toka

### 🏢 **Korporativne istovremene aplikacije**
- **Obrada velikog broja dokumenata**: Istovremena obrada više dokumenata
- **Analiza sadržaja u stvarnom vremenu**: Istovremena analiza dolaznih podataka
- **Optimizacija skupne obrade**: Maksimizirajte propusnost za velike operacije
- **Višestruka modalna analiza**: Paralelna obrada različitih vrsta sadržaja (tekst, slike, podaci)

## ⚙️ Preduvjeti i postavljanje

### 📦 **Potrebne ovisnosti**

Instalirajte Agent Framework s mogućnostima istovremenih radnih tokova:

```bash
pip install agent-framework -U
```

### 🔑 **Konfiguracija Microsoft Foundry**

Prijavite se putem Azure CLI (`az login`) kako bi se `AzureCliCredential` mogao autentificirati, a zatim postavite detalje vašeg Microsoft Foundry projekta.

**Postavljanje okoline (.env datoteka):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

**Razmatranja pri istovremenoj obradi:**
- **Ograničenja brzine**: Pratite Azure OpenAI ograničenja za istovremene zahtjeve
- **Korištenje resursa**: Razmotrite upotrebu memorije i CPU-a kod više istovremenih agenata
- **Rukovanje pogreškama**: Implementirajte robusno oporavljanje od pogrešaka za paralelne operacije

### 🏗️ **Arhitektura istovremenih radnih tokova**

```mermaid
graph TD
    A[Početak tijeka rada] --> B[Istovremeno izvođenje]
    B --> C[Skup agenata 1]
    B --> D[Skup agenata 2]
    B --> E[Skup agenata 3]
    C --> F[Skupljanje rezultata]
    D --> F
    E --> F
    F --> G[Završni rezultat]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Ključne prednosti:**
- **⚡ Performanse**: Značajno ubrzanje paralelnim izvođenjem
- **📈 Skalabilnost**: Rukovanje povećanim opterećenjima bez proporcionalnog povećanja vremena
- **🔄 Učinkovitost**: Bolja iskorištenost dostupnih računalnih resursa
- **🎯 Propusnost**: Obrada više posla u istom vremenu

## 🎨 **Obrasci dizajna istovremenih radnih tokova**

### 🔍 **Cjevovod istraživanja i analize**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Radni tok obrade podataka**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Cjevovod izrade sadržaja**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Višefazna obrada**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Poslovne pogodnosti performansi**

### ⚡ **Optimizacija propusnosti**
- **Paralelno izvođenje**: Više agenata koji rade istovremeno
- **Iskorištavanje resursa**: Maksimalna učinkovitost dostupnog kapaciteta AI modela
- **Smanjenje vremena**: Značajno smanjenje ukupnog vremena obrade
- **Skalabilna arhitektura**: Jednostavno dodajte više istovremenih agenata po potrebi

### 🛡️ **Pouzdanost i otpornost**
- **Otpornost na greške**: Pojedinačni padovi agenata ne zaustavljaju cijeli radni tok
- **Izolacija pogrešaka**: Problemi u jednom paralelnom ogranku ne utječu na druge
- **Postupno degradiranje**: Sustav nastavlja rad čak i s reduciranim kapacitetom agenata
- **Mehanizmi oporavka**: Automatsko ponavljanje i rukovanje pogreškama za neuspjele operacije

### 📊 **Praćenje i uvid**
- **Praćenje istovremenog izvođenja**: Pratite napredak svih paralelnih operacija
- **Metrički podaci performansi**: Mjerite ubrzanje i dobivene učinkovitosti
- **Analitika korištenja resursa**: Optimizirajte raspodjelu istovremenih agenata
- **Identifikacija uskih grla**: Pronađite i riješite ograničenja performansi

Izgradimo visokoučinkovite istovremene AI radne tokove! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
